# INF201 - DELIVERABLE 2

#### Heidi Christin Elvenes og Lisa Hoa Kieu Vo

#### Task 1: File ingestion and regular expressions

In [35]:
from pathlib import Path
import re, json

def read_log_file(file_path: Path) -> tuple[str, str]:
    file_path = Path(file_path)

    try:
        text = file_path.read_text(encoding = "utf-8")
        return text, "utf-8"
    except UnicodeDecodeError:
        text = file_path.read_text(encoding = "iso-8859-1")
        return text, "iso-8859-1"

PATTERN = re.compile(
    r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} "
    r"\d{2}:\d{2}:\d{2})\]\s+"
    r"\[(?P<register_id>[^\]]+)\]\s+"
    r"item=(?P<item>\S+)\s+"
    r"price=(?P<price>\d+(?:\.\d+)?)\s+"
    r"(?P<currency>EUR|NOK|USD)\s+"
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)"
)

def parse_log(text: str) -> list[dict]:
    records = []

    for line in text.splitlines():
        match = PATTERN.search(line)

        if match:
            record = match.groupdict()
            record["price"] = float(record["price"])
            records.append(record)

    return records

print(Path.cwd())

/Users/heidielvenes/Desktop/INF201/Deliverable2


##### Summary

The functions read the log files and extract transaction information using regular expressions. UTF-8 is tried first, with ISO-8859-1 as a fallback if decoding fails. The extracted data includes the timestamp, register ID, item, price, currency, and status. Invalid lines are skipped, and prices are converted to floats so they can be used in further calculations.

#### Task 2: Recursive utility functions

In [36]:
def find_log_files(directory: Path) -> list[Path]:
    directory = Path(directory)
    files = []

    for item in directory.iterdir():
        if item.is_dir():
            files.extend(find_log_files(item))
        elif item.is_file() and item.suffix == ".log":
            files.append(item)

    return files

def get_active_registers(node) -> set[str]:
    active = set()
    if isinstance(node, dict):
        if node.get("active") is True and "id" in node:
            active.add(node["id"])
        for value in node.values():
            active |= get_active_registers(value)
    
    elif isinstance(node, list):
        for element in node:
            active |= get_active_registers(element)
    return active

with open("store_config.json", encoding="utf-8") as f:
    config = json.load(f)

active_registers = get_active_registers(config)
print(active_registers)

{'POS_01', 'POS_02', 'POS_03'}


##### Summary

The `find_log_files()` function searches through the directory and its subdirectories to find all `.log` files. The `get_active_registers()` function searches through the nested configuration data and collects the IDs of active registers. These functions make it possible to find the relevant files and filter transactions based on which registers are active.

#### Task 3: Functions for cleaning and analysning data using NumPy

In [37]:
import numpy as np

def remove_price_outliers(prices) -> np.ndarray:
    prices = np.asarray(prices, dtype=float).ravel()
    outliers = np.abs(prices - prices.mean()) > 3 * prices.std()
    cleaned = prices.copy()
    if (~outliers).any():
        cleaned[outliers] = np.median(prices[~outliers])
    return cleaned

prices = np.array([3.0, 4.0, 4.5, 5.0, 3.5, 4.2, 3.8, 4.1, 3.9, 4.3, 4.0, 999.0])
print(remove_price_outliers(prices))

def compute_refund_ratio(
    sales_count: np.ndarray,
    refund_count: np.ndarray
) -> np.ndarray:
    return np.asarray(refund_count) / (
        np.asarray(sales_count) + 1e-6
    )

sales = np.array([100, 50, 0])
refunds = np.array([5, 10, 2])

np.set_printoptions(suppress = True, precision = 4)
print(compute_refund_ratio(sales, refunds))

[3.  4.  4.5 5.  3.5 4.2 3.8 4.1 3.9 4.3 4.  4. ]
[      0.05       0.2  2000000.  ]


##### Summary

NumPy is used to clean the price data and calculate refund ratios. The `remove_price_outliers()` function identifies prices that differ from the mean by more than three standard deviations and replaces them with the median of the remaining prices. The `compute_refund_ratio()` function calculates the ratio between refunds and successful sales for each item. 

#### Task 4: Data analysis using Pandas

In [38]:
import pandas as pd

def create_dataframe(records: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(records)
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.set_index("timestamp")
    df = df.sort_index()

    return df

def hourly_revenue(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby("item")
          .resample("1h")["price"]
          .sum()
          .unstack("item", fill_value = 0)
    )

def status_pivot(df: pd.DataFrame) -> pd.DataFrame:
    return pd.pivot_table(
        df,
        index = "item",
        columns = "status",
        values = "price",
        aggfunc = "sum",
        fill_value = 0
    )

def item_refund_ratios(df: pd.DataFrame) -> pd.DataFrame:
    counts = pd.crosstab(df["item"], df["status"])
    sales = counts["SUCCESS"].to_numpy()
    refunds = counts["REFUND"].to_numpy()
    # sales = counts.get(
    #    "SUCCESS", pd.Series(0, index=counts.index)
    #).to_numpy()

    #refunds = counts.get(
    #    "REFUND", pd.Series(0, index=counts.index)
    #).to_numpy()

    return pd.DataFrame({
        "item": counts.index,
        "sales_count": sales,
        "refund_count": refunds,
        "refund_ratio": compute_refund_ratio(sales, refunds)
    })

##### Summary

The timestamps are converted to datetime format, and the transactions are sorted chronologically. The functions then calculate hourly revenue, create a pivot table showing transaction values by item and status, and calculate refund ratios for each item.

#### Task 5: Create full workflow

##### Summary

The pipeline combines the functions from the previous tasks into one workflow. It finds the log files, loads the configuration, filters transactions from active registers, and extracts the transaction data. The prices are then cleaned, and NOK prices are multiplied by 10 to correct the currency. Then the pipeline generates the hourly revenue report, the status pivot table, and the refund ratio report. All four output files are then saved.

In [39]:

def run_pipeline(data_dir="2026", out_dir="."):
    data_dir = Path(data_dir)
    out_dir = Path(out_dir)

    # Finner loggfiler og aktive registre
    log_files = find_log_files(data_dir)

    with open("store_config.json", encoding="utf-8") as f:
        active_registers = get_active_registers(json.load(f))
    print(f"Fant {len(log_files)} loggfiler. Aktive registre: {sorted(active_registers)}")

    # Leser og parser
    records = []
    for path in log_files:
        text, encoding = read_log_file(path)
        parsed = parse_log(text)
        print(f"{path}: {len(parsed)} transaksjoner ({encoding})")
        records.extend(parsed)

    # Beholder kun aktive registre
    records = [r for r in records if r["register_id"] in active_registers]

    # DataFrame
    df = create_dataframe(records)

    # Outlier-rensing
    df["price"] = remove_price_outliers(df["price"].to_numpy())

    # Valutakorrigering (NOK registrert i EUR-skala)
    df.loc[df["currency"] == "NOK", "price"] *= 10.0

    # Rapporter
    hourly = hourly_revenue(df)
    pivot = status_pivot(df)
    ratios = item_refund_ratios(df)

    # Lagre
    df.to_csv(out_dir / "cleaned_transactions.csv")

    hourly_out = hourly.copy()
    hourly_out.index = hourly_out.index.astype(str)
    hourly_out.to_json(out_dir / "hourly_revenue.json", indent=2)

    pivot.to_csv(out_dir / "status_pivot_table.csv")
    ratios.to_csv(out_dir / "item_refund_ratios.csv", index=False)

    return df, hourly, pivot, ratios

df, hourly, pivot, ratios = run_pipeline()
display(pivot)
display(ratios)

Fant 9 loggfiler. Aktive registre: ['POS_01', 'POS_02', 'POS_03']
2026/science_building/POS_01/register_sales.log: 80 transaksjoner (utf-8)
2026/science_building/POS_03/register_sales.log: 80 transaksjoner (utf-8)
2026/science_building/POS_02/register_sales.log: 80 transaksjoner (utf-8)
2026/student_union/POS_01/register_sales.log: 80 transaksjoner (utf-8)
2026/student_union/POS_03/register_sales.log: 80 transaksjoner (utf-8)
2026/student_union/POS_02/register_sales.log: 80 transaksjoner (utf-8)
2026/library_cafe/POS_01/register_sales.log: 80 transaksjoner (utf-8)
2026/library_cafe/POS_03/register_sales.log: 80 transaksjoner (utf-8)
2026/library_cafe/POS_02/register_sales.log: 80 transaksjoner (utf-8)


status,CANCELLED,REFUND,SUCCESS
item,,,
Cappuccino,294.03,401.81,1304.15
Croissant,301.16,293.73,1382.49
Espresso,241.66,273.74,1286.15
Latte,248.27,290.66,1211.67
Muffin,483.15,303.29,1225.79
Sandwich,772.86,466.45,1874.31


,item,sales_count,refund_count,refund_ratio
0,Cappuccino,96,18,0.187500
1,Croissant,87,16,0.183908
2,Espresso,97,20,0.206186
3,Latte,64,17,0.265625
4,Muffin,77,22,0.285714
5,Sandwich,72,17,0.236111
